In [1]:
# %% 0 - percorsi (STESSI NOMI di 09C_06/09C_07, cosi' le celle di 09C_07 girano senza modifiche)
import sys, json, shutil, subprocess
from pathlib import Path
import pandas as pd
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark
from src.evaluation import distance_presets as dp
from src.evaluation import colosseum_controls as cc
from src.evaluation.json_ingest import ingest_chunks_json, load_tags_into_sidecar
pd.set_option("display.width", 250, "display.max_columns", 30)
 
COLLECTION = "colosseum_bench"                      # = COLLECTION_NAME in src/config.py (invariato)
Q_OLD = project_root / "data/queries/colosseum"     # file di 09C_06 (sorgente)
QDIR = project_root / "data/queries/colosseum_v2"   # file nuovi
sidecar_path = project_root / "data/processed/colosseum/colosseum_v2.json"
base_sidecar = sidecar_path.with_name("colosseum_v2_base.json")
chunks_path, queries_path = QDIR / "chunks_colosseum_final.json", QDIR / "eval_queries.json"
pred_path, norm_path, gold_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json", QDIR / "gold_chunks.json"
OUT_DIR = project_root / "data/benchmark/colosseum_v2"
TOP_J, TOP_K, TOP_N = 5, 15, 7                      # regime di partenza: lo sweep di 09C_07 (cella 4) dice se cambiarlo

In [2]:
# %% 1 - genera i file v2 (monumento sui chunk e come conv_tag di OGNI query). Stampa la tabella: RIVEDI i conv_tag assegnati.
# Regole: monumento = quello del PRIMO chunk gold; conv_tag = [monumento, aspetto] per le ambigue (tranne AMB_04/05, vedi MULTI_CONV_EXCLUDE in add_monument_tags.py).
src = project_root / "src/evaluation"
r = subprocess.run([sys.executable, str(src / "add_monument_tags.py"), "--chunks", str(Q_OLD / "chunks_colosseum_final.json"),
                    "--out", str(QDIR), "--make_queries", str(src / "make_queries.py")], capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])

predeterminate: ['AMB_01', 'AMB_02', 'AMB_05', 'AMB_11', 'MUL_02', 'MUL_09', 'MUL_11', 'MUL_12', 'SPE_02', 'SPE_03', 'SPE_04', 'SPE_14']
chunk per monumento: {'Colosseum': 68, 'DomusAurea': 20, 'ColossusOfNero': 4, 'LudusMagnus': 11, 'ArchOfConstantine': 17}
monumento (primo conv_tag): {'Colosseum': 31, 'ColossusOfNero': 2, 'DomusAurea': 4, 'LudusMagnus': 3, 'ArchOfConstantine': 2}
query con piu' conv tag: 12 di 42

id       conv_tag                            tag manuali                  pool cat.1  gold su altri monumenti
AMB_01   Colosseum+Conservation                                            11      -
AMB_02   Colosseum+Archaeology                                             19      -
AMB_03   Colosseum+Archaeology                                             19      -
AMB_04   ColossusOfNero                      History                        4      DomusAurea
AMB_05   Colosseum                           Culture                       68      ColossusOfNero
AMB_06   Colosseum+Cons

In [3]:
# %% 2 - ingestione (collezione + tag nel sidecar v2) e handler/bench. UNA volta; poi gli stessi file servono a tutte le celle.
sidecar_path.parent.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)
ingest_chunks_json(chunks_path, COLLECTION, sidecar_path).release_path()      # il sidecar v2 e' nuovo: nessuna distanza ne' predq ereditata
def make_bench(sp, h=None):
    h = h or PipelineHandler(sidecar_path=sp, queries_path=queries_path)
    return RagasBenchmark(h, predetermined_path=pred_path, normal_path=norm_path, out_dir=OUT_DIR,
                          top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
bench = make_bench(sidecar_path); handler = bench.h; sm = handler.sidecar_manager
allq = json.load(open(queries_path, encoding="utf-8")); gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
TAGS = sorted({t for c in chunks.values() for t in c["tags"]})
assert all(cc.conv_of(q) and cc.conv_of(q)[0] in cc.MONUMENT_TAGS for q in allq), "ogni query deve avere un monumento come primo conv_tag"
assert all(set(cc.conv_of(q)[1:]) <= set(TAGS) - set(cc.MONUMENT_TAGS) and set(q["tags"]) <= set(TAGS) - set(cc.MONUMENT_TAGS) for q in allq)
print("query con piu' conv tag:", sum(len(cc.conv_of(q)) > 1 for q in allq), "di", len(allq))
print({k: v["count"] for k, v in sm.get_global_tags().items()})

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


<<! File sidecar ripristinato allo stato iniziale !>>
!>> 120 chunk in 'colosseum_bench' (20.8s); tag nel sidecar: ['Colosseum', 'History', 'Architecture', 'Culture', 'Spectacles', 'Conservation', 'Archaeology', 'DomusAurea', 'ColossusOfNero', 'LudusMagnus', 'ArchOfConstantine']
query con piu' conv tag: 12 di 42
{'Colosseum': 68, 'History': 48, 'Architecture': 48, 'Culture': 19, 'Spectacles': 28, 'Conservation': 14, 'Archaeology': 29, 'DomusAurea': 20, 'ColossusOfNero': 4, 'LudusMagnus': 11, 'ArchOfConstantine': 17}


In [4]:
# %% 3 - predq + similarita' + backup base (tag + predq, nessuna distanza)
print(len(bench.register_predetermined("pred")), "predq registrate")
display(bench.check_similarity(warn_above=0.80).head(6))
shutil.copy(sidecar_path, base_sidecar)

12 predq registrate


!> 1 query normali troppo simili a una predeterminata: ['MUL_01']


,normal_id,max_sim,closest_pred
20,MUL_01,0.826329,MUL_09
14,SPE_08,0.783214,AMB_05
25,MUL_07,0.777352,AMB_05
9,AMB_14,0.775193,AMB_05
19,SPE_13,0.761778,MUL_12
21,MUL_03,0.752021,SPE_02


PosixPath('/home/jovyan/tesi_graphrag/data/processed/colosseum/colosseum_v2_base.json')

In [5]:
# %% 4 - versione ESPERTA: stesso piano di distanze di 09C_06 (dipende da gold e chunk, non dai tag) + retrieval + risposte
plan_expert_path = Q_OLD / "distance_plan_colosseum_expert.json"                # il file che mi hai mandato
plan_expert = dp.load_plan(plan_expert_path)
assert {k.replace("predq_", "") for k in plan_expert["predq"]} == {q["id"] for q in json.load(open(pred_path))}, "predq del piano != predeterminate v2"
print(dp.apply_plan(sm, plan_expert))
genA = bench.generate("pilot_expert_v2", with_answers=False)       # retrieval soltanto (~1 min): prima di spendere LLM
display(bench.retrieval_summary("pilot_expert_v2"))
META = ["run_id", "query_id", "category", "subtype", "is_predetermined", "config"]; GOLD = ["gold_hit", "gold_recall", "gold_precision", "gold_mrr"]
mA = genA[META + GOLD]; normal = ~mA.is_predetermined
for lab, sub in (("NORMALI", normal), ("AMBIGUE normali", normal & (mA.category == "ambiguous"))):
    r = bench.paired_vs_baseline(mA, subset=sub); print("---", lab); display(r[r.metric.isin(["gold_recall", "gold_mrr"])])
# gen = bench.generate("post_edit_expert_v2")                       # con risposte: SOLO dopo aver scelto (top_k, top_j) con 09C_07 cella 4

<<| Batch adiacenze query pre-determinate salvato: 99 elementi processati |>>
<<| Batch adiacenze standard salvato: 499 elementi processati |>>
{'predq_edits': 99, 'chunk_pairs': 499}


TypeError: unhashable type: 'list'

In [ ]:
# %% 5 - versione UTENTE MEDIO a due assi (aspetto: 30% mancanti/15% sbagliati/solo primo; monumento: 10% mancante/5% sbagliato)
KEEP, SEED = 0.6, 7
plan_user = dp.degrade_plan(plan_expert, keep=KEEP, seed=SEED)
topical = [t for t in TAGS if t not in cc.MONUMENT_TAGS]
kt, km = cc.VARIANTS["combinato_user"]
chunks_user, log = cc.degrade_two_axes(list(chunks.values()), topical, topical=kt, monument=km, seed=SEED)
print(log)
user_sidecar = sidecar_path.with_name("colosseum_v2_user.json"); shutil.copy(base_sidecar, user_sidecar)
load_tags_into_sidecar(chunks_user, user_sidecar).release_path()
bench_u = make_bench(user_sidecar); print(dp.apply_plan(bench_u.h.sidecar_manager, plan_user))
# gen_u = bench_u.generate("post_edit_user_v2")                      # con risposte, come sopra

## Controllo celle
Commenti per funzionamento

In [ ]:
# %% 0 - setup
import importlib, json, shutil
import pandas as pd
from src.evaluation import colosseum_controls as cc
importlib.reload(cc)
pd.set_option("display.width", 250, "display.max_columns", 30)
gold = json.load(open(gold_path, encoding="utf-8"))
chunks = {c["chunk_id"]: c for c in json.load(open(chunks_path, encoding="utf-8"))}
TAGS = sorted({t for c in chunks.values() for t in c["tags"]})
exp_gen = bench.load_generation("pilot_expert_v2")          # riferimento: retrieval esperto v2 con tag giusti (cella 4 di 09C_08), stesso regime (top_k=15, top_j=5)
print(len(allq), "query;", len(chunks), "chunk;", TAGS)
assert set(cc.MONUMENT_TAGS) <= set(TAGS), "dati v1: esegui prima 09C_08 (celle 0-4) in questo kernel"
assert "colosseum_v2" in str(chunks_path), f"chunks_path punta ai file v1: {chunks_path}"

In [ ]:
# %% 1 - CONTROLLO NEGATIVO su entrambi gli assi (ca. 2 min). Atteso: il recall CALA rispetto al run con tag giusti.
#   monument: conv_tag = ALTRO monumento (aspetto giusto) | aspect: conv_tag giusto, aspetto (tag manuale) sbagliato
for mode in ("monument", "aspect"):
    wq = cc.wrong_tag_queries(bench.queries, gold, chunks, TAGS, seed=0, mode=mode)
    wrong = cc.run(bench, f"ctrl_wrong_{mode}", queries=wq)
    print(f"=== {mode}: quota dei top-J che porta il tag sbagliato usato"); display(cc.wrong_tag_leak(wrong, wq, chunks, mode=mode).to_frame().T)
    t = cc.paired_between_runs(exp_gen, wrong, subsets=("normali", "ambigue normali"))    # B - A = tag sbagliato - tag giusto
    display(t[t.metric == "gold_recall"].drop(columns="metric"))

In [ ]:
# %% 2 - DUE LETTURE: stessa query, tag A / tag B (bozza in cc.TWO_READINGS: RIVEDILA leggendo i chunk prima di fidarti)
pairs = cc.TWO_READINGS
cc.validate_two_readings(pairs, gold, chunks)
for qid, p in pairs.items():
    q = next(q for q in allq if q["id"] == qid)
    print(f"{qid}: {q['query']}\n   A={q['expected_tag']} {gold[qid]}\n   B={p['tag_b']} {p['gold_b']}")
tq = cc.two_readings_queries(bench.queries, pairs)
tr = cc.run(bench, "ctrl_tworeadings", queries=tq)
rep, det = cc.two_readings_report(tr, pairs, gold)
display(rep)         # specificita' = R_own - R_other: C0 = 0 per costruzione; con i tag deve diventare > 0
display(det[det.config.isin(["C0_baseline", "C1_tags", "C4_dist_tags_joinconv"])].pivot_table(index=["query", "lettura"], columns="config", values=["R_own", "R_other"]).round(2))

In [ ]:
# %% 3 - DEGRADAZIONE TAG SEPARATA (6 varianti x 3 seed; sidecar BASE = tag + predq, nessuna distanza). ~15 min.
from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark
def make_bench(sp):
    h = PipelineHandler(sidecar_path=sp, queries_path=queries_path)
    return RagasBenchmark(h, predetermined_path=pred_path, normal_path=norm_path, out_dir=OUT_DIR,
                          top_j=TOP_J, top_k=TOP_K, top_n=TOP_N, gold_path=gold_path)
deg = cc.degradation_study(chunks, TAGS, base_sidecar, work_dir=sidecar_path.parent / "deg", make_bench=make_bench, seeds=(0, 1, 2))
deg.to_csv(OUT_DIR / "ctrl_degradation_retrieval.csv", index=False)
display(cc.summarize_vs_c0(deg))     # differenza con C0 (media|std sui seed): quale errore dei tag fa piu' danno?

In [ ]:
# %% 4 - DISTANZE: sidecar CON distanze (esperto) contro SENZA distanze (stessi tag), per regime (top_k, top_j). ~15 min.
# Regimi: (15,5) (10,5) (7,5) = pochi posti liberi; (5,8) (5,10) = posti liberi sempre >= 3 (i 5 vettoriali fanno da seme, il grafo riempie il resto).
# C0 e' sempre il vettoriale puro a top_j chunk (run_regime), cosi' il confronto e' alla pari.
nodist_sidecar = sidecar_path.with_name("colosseum_benchmark_nodist.json")
shutil.copy(base_sidecar, nodist_sidecar)                 # base = tag esperti + predq, nessuna adiacenza
bench_nd = make_bench(nodist_sidecar)
tab, gens = cc.distance_ablation(bench, bench_nd)
tab.to_csv(OUT_DIR / "ctrl_distance_ablation.csv", index=False)
view = tab[(tab.metric == "gold_recall")].drop(columns="metric")
print("EFFETTO DELLE DISTANZE (con - senza), gold_recall; quota_query_cambiate = normali in cui cambiano i top-J")
display(view[view.config != "C0_baseline"].sort_values(["subset", "top_k", "top_j", "config"]))
print("config contro C0 (diff_vs_C0), SENZA distanze (side=0) e CON distanze (side=1):")
display(cc.vs_c0_by_topk(gens, side=0)); display(cc.vs_c0_by_topk(gens, side=1))
print("quota di query con contesto CORTO (< top_j chunk: categoria 1 vuota, distanze non partite), regime (5,8), con distanze:")
display(cc.short_context_quota(gens[(5, 8)][1], 8))